# Part 3 – Ensemble models, tuning, final model & predictions (Abhiram)
Run from inside `1-challenge/`. Models: Random Forest, Gradient Boosting (required) + HistGradientBoosting (extra). Same 5-fold CV / RMSE protocol as the rest of the team notebook.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import KFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

SEED = 42
train = pd.read_csv("data/challenge1_train.csv")
test  = pd.read_csv("data/challenge1_test.csv")
print(train.shape, test.shape)
train.head()

(7398, 10) (2467, 9)


,ID,price,bathrooms,bedrooms,square_feet,latitude,longitude,category,has_photo,state
0,2,925,1.0,0,116,47.6160,-122.3275,housing/rent/apartment,Thumbnail,WA
1,3,2475,1.0,0,130,40.7629,-73.9885,housing/rent/apartment,Thumbnail,NY
2,5,1695,1.0,0,190,37.7599,-122.4379,housing/rent/apartment,Thumbnail,CA
3,6,1560,1.0,1,200,35.0847,-77.0609,housing/rent/apartment,Thumbnail,NC
4,7,1560,1.0,1,200,35.0960,-77.0272,housing/rent/apartment,Thumbnail,NC


## 2. Preprocessing & validation strategy
* **Preprocessing lives inside sklearn Pipelines**, so scaling/encoding are fit only on the training folds (no leakage).
* Linear model: standardize numerics + one-hot categoricals. Tree models: no scaling needed.
* **Validation: 5-fold shuffled KFold (seed 42) on the training data only**, used for every model, every grid search and the final choice. Metric: RMSE on the original dollar scale. We report train RMSE vs CV RMSE to diagnose over/underfitting.
* Test set is never touched until the final prediction.

In [2]:
X = train.drop(columns=["ID","price","category"]); y = train["price"]
X_test = test.drop(columns=["ID","category"])
num = ["bathrooms","bedrooms","square_feet","latitude","longitude"]; cat = ["has_photo","state"]
cv = KFold(5, shuffle=True, random_state=SEED)

def prep(scale):
    return ColumnTransformer([("num", StandardScaler() if scale else "passthrough", num),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])

def evaluate(name, model, X=X):
    r = cross_validate(model, X, y, cv=cv, scoring="neg_root_mean_squared_error", return_train_score=True, n_jobs=-1)
    return dict(model=name, train_RMSE=-r["train_score"].mean(), cv_RMSE=-r["test_score"].mean(), cv_std=r["test_score"].std())

## Baselines (default hyperparameters)

In [3]:
base = {
 "Random Forest":     Pipeline([("p",prep(False)),("m",RandomForestRegressor(n_estimators=200,random_state=SEED,n_jobs=-1))]),
 "Gradient Boosting": Pipeline([("p",prep(False)),("m",GradientBoostingRegressor(random_state=SEED))]),
 "Hist Gradient Boosting (extra)": Pipeline([("p",prep(False)),("m",HistGradientBoostingRegressor(random_state=SEED))]),
}
res = pd.DataFrame([evaluate(k,v) for k,v in base.items()]).round(1); res

,model,train_RMSE,cv_RMSE,cv_std
0,Random Forest,161.2,423.7,33.3
1,Gradient Boosting,406.7,461.4,27.7
2,Hist Gradient Boosting (extra),317.6,409.1,26.1


## Feature engineering + log target
Row-wise features inside the pipeline (no leakage): `log_sqft`, `total_rooms`, `sqft_per_bed`, `bath_per_bed`. Target `log1p(price)`, inverted with `expm1`; RMSE stays in dollars.

In [4]:
def add_feats(d):
    d = d.copy()
    d["log_sqft"] = np.log1p(d.square_feet)
    d["total_rooms"] = d.bedrooms + d.bathrooms
    d["sqft_per_bed"] = d.square_feet / (d.bedrooms + 1)
    d["bath_per_bed"] = d.bathrooms / (d.bedrooms + 1)
    return d
FE = FunctionTransformer(add_feats)
num2 = num + ["log_sqft","total_rooms","sqft_per_bed","bath_per_bed"]
def prep2(scale):
    return ColumnTransformer([("num", StandardScaler() if scale else "passthrough", num2),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])
def pipe(model, scale=False, logy=True):
    p = Pipeline([("fe",FE),("p",prep2(scale)),("m",model)])
    return TransformedTargetRegressor(p, func=np.log1p, inverse_func=np.expm1) if logy else p

## Grid Search (same 5-fold CV)

In [5]:
grids = {
 "Random Forest": (pipe(RandomForestRegressor(n_estimators=200,random_state=SEED,n_jobs=-1)),
     {"regressor__m__max_depth":[None,20],"regressor__m__min_samples_leaf":[1,3],"regressor__m__max_features":[0.5,1.0]}),
 "Gradient Boosting": (pipe(GradientBoostingRegressor(random_state=SEED)),
     {"regressor__m__n_estimators":[300,600],"regressor__m__learning_rate":[0.03,0.08],"regressor__m__max_depth":[3,5],"regressor__m__subsample":[0.8]}),
 "Hist Gradient Boosting": (pipe(HistGradientBoostingRegressor(random_state=SEED)),
     {"regressor__m__learning_rate":[0.03,0.06,0.1],"regressor__m__max_iter":[300,600],"regressor__m__max_leaf_nodes":[15,31,63],"regressor__m__l2_regularization":[0,1]}),
}
best, rows = {}, []
for name,(est,g) in grids.items():
    gs = GridSearchCV(est, g, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1, return_train_score=True).fit(X,y)
    best[name]=gs.best_estimator_; i=gs.best_index_
    rows.append(dict(model=name, train_RMSE=-gs.cv_results_["mean_train_score"][i], cv_RMSE=-gs.best_score_, cv_std=gs.cv_results_["std_test_score"][i], best_params={k.split("__")[-1]:v for k,v in gs.best_params_.items()}))
tuned = pd.DataFrame(rows).sort_values("cv_RMSE"); tuned.round(1)

,model,train_RMSE,cv_RMSE,cv_std,best_params
1,Gradient Boosting,191.1,390.7,28.9,"{'learning_rate': 0.08, 'max_depth': 5, 'n_est..."
2,Hist Gradient Boosting,262.2,396.4,25.2,"{'l2_regularization': 0, 'learning_rate': 0.1,..."
0,Random Forest,177.2,420.9,35.5,"{'max_depth': None, 'max_features': 1.0, 'min_..."


## Final model selection

In [6]:
FINAL_NAME = tuned.iloc[0].model
final = best[FINAL_NAME]
print("Final model:", FINAL_NAME, "| CV RMSE =", round(tuned.iloc[0].cv_RMSE,1))
# Quick interpretability check: permutation importance on a held-out fold
from sklearn.inspection import permutation_importance
tr, va = next(iter(cv.split(X)))
m = __import__("sklearn").base.clone(final).fit(X.iloc[tr], y.iloc[tr])
pi = permutation_importance(m, X.iloc[va], y.iloc[va], scoring="neg_root_mean_squared_error", n_repeats=5, random_state=SEED)
pd.Series(pi.importances_mean, X.columns).sort_values(ascending=False).round(1)

Final model: Gradient Boosting | CV RMSE = 390.7


longitude      342.2
square_feet    254.8
latitude       225.7
state          156.7
bathrooms       58.1
bedrooms        21.2
has_photo        1.6
dtype: float64

### Justification
Lowest CV RMSE among the ensembles; train–CV gap is moderate (not a memorizer like the untuned tree, not underfit like linear regression); trains in seconds. Less interpretable than linear regression, but permutation importance shows sensible drivers (location, size, bathrooms).

## Final fit on all training data → predictions.csv

In [7]:
final.fit(X, y)
pred = final.predict(X_test)
out = pd.DataFrame({"ID": test.ID, "price": np.round(pred, 2)})
assert len(out)==len(test)==2467 and out.ID.is_unique and out.price.notna().all()
out.to_csv("predictions.csv", index=False)
print(out.price.describe().round(1)); out.head()

count    2467.0
mean     1444.7
std       757.5
min       453.6
25%       983.2
50%      1240.2
75%      1640.7
max      8496.6
Name: price, dtype: float64


,ID,price
0,1,1378.68
1,4,1659.95
2,9,921.06
3,11,814.97
4,13,633.61
